In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Setup Paths and Parameters (with changes for ResNet) ---

data_path = 'mnist-pngs-main/' 
train_path = data_path + 'train'
test_path = data_path + 'test'

# --- CRITICAL CHANGES FOR RESNET50 ---
# ResNet50 was trained on 224x224 images. We must resize our images to match.
BATCH_SIZE = 32
IMG_HEIGHT = 224 
IMG_WIDTH = 224

# --- 2. Load the Datasets ---
# We will still load them as grayscale initially and handle the color channel
# conversion in the preprocessing step.

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale', # Load as grayscale first
    image_size=(IMG_HEIGHT, IMG_WIDTH), # Resize to 224x224
    batch_size=BATCH_SIZE
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    labels='inferred',
    label_mode='int',
    color_mode='grayscale', # Load as grayscale first
    image_size=(IMG_HEIGHT, IMG_WIDTH), # Resize to 224x224
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Class names: {class_names}")

# --- 3. Prepare the Dataset for ResNet50 Training (Major Changes) ---

# This function is now more complex. It needs to do two things:
# 1. Convert our 1-channel grayscale image to a 3-channel image, as ResNet expects RGB.
# 2. Apply the specific preprocessing (pixel scaling/normalization) that ResNet50 was originally trained with.

def preprocess_for_resnet(image, label):
    # Convert grayscale to RGB by repeating the channel.
    image = tf.image.grayscale_to_rgb(image)
    # Use the official ResNet50 preprocessing function.
    # This function scales pixel values to the range expected by the model.
    return tf.keras.applications.resnet50.preprocess_input(image), label

# Apply this new preprocessing function to our datasets
train_dataset = train_dataset.map(preprocess_for_resnet)
test_dataset = test_dataset.map(preprocess_for_resnet)

# Optimize performance with caching and prefetching (same as before)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

print("\n--- Inspecting One Preprocessed Batch ---")
for images, labels in train_dataset.take(1):
    print(f"Images batch shape after preprocessing: {images.shape}") # Should be (32, 224, 224, 3)
    print(f"Labels batch shape: {labels.shape}")
    break # We just need to see the shape of one batch

# --- 4. Build the Transfer Learning Model ---

# Step 1: Load the pre-trained ResNet50 model as our base.
# - `weights='imagenet'`: Loads weights pre-trained on the ImageNet dataset.
# - `include_top=False`: This is CRITICAL. It removes the original final dense layer 
#   that was used to classify 1000 ImageNet classes. We will add our own.
# - `input_shape`: The shape of our preprocessed images.
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Step 2: Freeze the base model.
# We freeze the layers of the base_model so their weights are NOT updated
# during training. We are treating it as a fixed feature extractor.
base_model.trainable = False

# Step 3: Create our new model by adding our own classification layers on top.
# We use the Keras Functional API for this.
inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
# The base model will extract features from the input. 
# `training=False` ensures it runs in inference mode (e.g., for BatchNorm layers).
x = base_model(inputs, training=False)
# We need to flatten the feature map output from the base model before
# feeding it to our dense layers. GlobalAveragePooling2D is often preferred.
x = tf.keras.layers.GlobalAveragePooling2D()(x)
# Add a dropout layer for regularization to prevent overfitting.
x = tf.keras.layers.Dropout(0.2)(x)
# Add our final classification layer with 10 neurons (for 10 digits)
# and a softmax activation for probability scores.
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

# Combine the inputs and outputs to create the final model
model = tf.keras.Model(inputs, outputs)

# Step 4: Compile the model.
# The compile and fit steps are very similar to before.
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print("\n--- Model Summary ---")
model.summary()
print("\nNotice how the vast majority of parameters are 'Non-trainable'.")
print("These are the frozen weights from the pre-trained ResNet50 model.")

# --- 5. Train the Model ---
print("\n--- Starting Model Training (only training the top classification layer) ---")
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5  # 5 epochs is enough to see good results
)

# --- 6. Evaluate the Model ---
print("\n--- Evaluating Model on Test Data ---")
loss, accuracy = model.evaluate(test_dataset)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

# --- 7. (Optional) Visualize Predictions ---
print("\n--- Making predictions on a test batch ---")
for test_images, test_labels in test_dataset.take(1):
    # The preprocessed images are not suitable for direct viewing with plt.imshow
    # as their pixel values are not in the [0, 255] or [0, 1] range.
    
    # Make predictions
    predictions = model.predict(test_images)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Since we can't easily show the images, we'll just compare predicted vs true labels
    print("Predicted classes:", predicted_classes)
    print("True labels:      ", test_labels.numpy())
    break